# Unit Tests — `config.py`

Comprehensive tests for all shared functions in `app/config.py`.
Mocks Streamlit and Databricks SDK so tests run anywhere.

| Suite | Functions Tested |
| --- | --- |
| `TestToBold` | `to_bold()` – Unicode bold conversion |
| `TestIsNa` | `is_na()` – NA/missing detection |
| `TestValidateMandatoryCols` | `validate_mandatory_cols()` |
| `TestNormValue` | `norm_value()` – value normalization |
| `TestSafeTupleEq` | `safe_tuple_eq()` – NA-safe comparison |
| `TestToAuditStr` | `to_audit_str()` – audit string |
| `TestBuildCompositeWhere` | `build_composite_where()` – WHERE clauses |
| `TestBuildCompositeSet` | `build_composite_set()` – SET clauses |
| `TestAdminGetVisibleColumns` | `admin_get_visible_columns()` |
| `TestGetDisabledColumns` | `get_disabled_columns()` – per-user |
| `TestEditorGroupSCIM` | `is_user_in_editor_group()` – AAD group |
| `TestDropdownOptions` | `dropdown_options()` |
| `TestGenerateTestId` | `generate_test_id()` – SP test ID |
| `TestBuildAuditEvents` | `build_audit_events()` |
| `TestConstants` | Config constants validation |
| `TestEnsureSessionId` | `ensure_session_id()` |

In [0]:
# ============================================================
# Mock Streamlit + Databricks SDK BEFORE importing config
# ============================================================

import os, sys, json, importlib
import unittest
from unittest.mock import patch, MagicMock
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# ── Streamlit mock ────────────────────────────────────────
mock_st = MagicMock()
mock_st.cache_data = lambda **kw: (lambda f: f)
mock_st.cache_resource = lambda **kw: (lambda f: f)
mock_st.session_state = {}
mock_st.context.headers = {}
sys.modules['streamlit'] = mock_st

# ── Databricks SDK mock ───────────────────────────────────
sys.modules['databricks'] = MagicMock()
sys.modules['databricks.sql'] = MagicMock()
sys.modules['databricks.sdk'] = MagicMock()
sys.modules['databricks.sdk.core'] = MagicMock()

mock_certifi = MagicMock()
mock_certifi.where.return_value = '/tmp/fake-ca.pem'
sys.modules['certifi'] = mock_certifi
sys.modules['dotenv'] = MagicMock()

# ── Env vars ─────────────────────────────────────────────
os.environ['DATABRICKS_WAREHOUSE_ID'] = 'test_warehouse_123'
os.environ['DATABRICKS_HOST'] = 'https://test.databricks.com'

mock_cfg = MagicMock()
mock_cfg.host = 'https://test.databricks.com'
mock_cfg.warehouse_id = 'test_warehouse_123'
sys.modules['databricks.sdk.core'].Config = lambda: mock_cfg

# ── Import config ────────────────────────────────────────
sys.path.insert(0, '/Workspace/Users/pavankumar.penikalapati@external.danone.com/sn-ux-db-table-editor/app')
if 'config' in sys.modules:
    del sys.modules['config']
import config
importlib.reload(config)
print('✅ config.py imported successfully')

In [0]:
# ============================================================
# TEST: to_bold() – Unicode bold conversion
# ============================================================

class TestToBold(unittest.TestCase):
    def test_lowercase(self):
        self.assertEqual(config.to_bold('abc'), '𝗮𝗯𝗰')

    def test_uppercase(self):
        self.assertEqual(config.to_bold('ABC'), '𝗔𝗕𝗖')

    def test_digits(self):
        self.assertEqual(config.to_bold('123'), '𝟭𝟮𝟯')

    def test_mixed(self):
        result = config.to_bold('row_id')
        self.assertIn('𝗿', result)
        self.assertIn('_', result)

    def test_empty_string(self):
        self.assertEqual(config.to_bold(''), '')

    def test_special_chars_passthrough(self):
        self.assertEqual(config.to_bold('_-. '), '_-. ')

    def test_none_input(self):
        result = config.to_bold(None)
        self.assertEqual(result, '𝗡𝗼𝗻𝗲')

In [0]:
# ============================================================
# TEST: is_na() – NA/missing value detection
# ============================================================

class TestIsNa(unittest.TestCase):
    def test_none(self):       self.assertTrue(config.is_na(None))
    def test_pd_na(self):      self.assertTrue(config.is_na(pd.NA))
    def test_np_nan(self):     self.assertTrue(config.is_na(np.nan))
    def test_float_nan(self):  self.assertTrue(config.is_na(float('nan')))
    def test_empty_str(self):  self.assertTrue(config.is_na(''))
    def test_whitespace(self): self.assertTrue(config.is_na('   '))
    def test_pd_nat(self):     self.assertTrue(config.is_na(pd.NaT))
    def test_valid_str(self):  self.assertFalse(config.is_na('hello'))
    def test_zero(self):       self.assertFalse(config.is_na(0))
    def test_false(self):      self.assertFalse(config.is_na(False))
    def test_number(self):     self.assertFalse(config.is_na(42))

In [0]:
# ============================================================
# TEST: validate_mandatory_cols()
# ============================================================

class TestValidateMandatoryCols(unittest.TestCase):
    def test_all_filled(self):
        df = pd.DataFrame({'row_id': [1, 2], 'year': [2024, 2025], 'brand': ['A', 'B']})
        self.assertEqual(config.validate_mandatory_cols(df, ['year', 'brand']), [])

    def test_missing_values(self):
        df = pd.DataFrame({'row_id': [1, 2], 'year': [2024, None], 'brand': ['A', '']})
        errors = config.validate_mandatory_cols(df, ['year', 'brand'])
        self.assertEqual(len(errors), 2)

    def test_column_not_in_df(self):
        df = pd.DataFrame({'row_id': [1], 'year': [2024]})
        self.assertEqual(config.validate_mandatory_cols(df, ['nonexistent']), [])

    def test_all_empty(self):
        df = pd.DataFrame({'row_id': [1, 2], 'col': [None, '']})
        self.assertEqual(len(config.validate_mandatory_cols(df, ['col'])), 2)

    def test_pd_na_values(self):
        df = pd.DataFrame({'row_id': [1], 'col': pd.array([pd.NA], dtype=pd.StringDtype())})
        self.assertEqual(len(config.validate_mandatory_cols(df, ['col'])), 1)

In [0]:
# ============================================================
# TEST: norm_value() – value normalization for grouping
# ============================================================

class TestNormValue(unittest.TestCase):
    def test_none(self):              self.assertEqual(config.norm_value(None), '')
    def test_pd_na(self):             self.assertEqual(config.norm_value(pd.NA), '')
    def test_np_nan(self):            self.assertEqual(config.norm_value(np.nan), '')
    def test_string_strip(self):      self.assertEqual(config.norm_value('  hi  '), 'hi')
    def test_year_int(self):          self.assertEqual(config.norm_value(2024, 'test_year'), '2024')
    def test_year_float(self):        self.assertEqual(config.norm_value(2024.0, 'test_year'), '2024')
    def test_year_string(self):       self.assertEqual(config.norm_value('2024', 'test_year'), '2024')
    def test_year_none(self):         self.assertEqual(config.norm_value(None, 'test_year'), '')
    def test_year_invalid(self):      self.assertEqual(config.norm_value('abc', 'test_year'), '')
    def test_number(self):            self.assertEqual(config.norm_value(42), '42')

In [0]:
# ============================================================
# TEST: safe_tuple_eq() – NA-safe tuple comparison
# ============================================================

class TestSafeTupleEq(unittest.TestCase):
    def test_equal(self):          self.assertTrue(config.safe_tuple_eq(('a', 'b'), ('a', 'b')))
    def test_unequal(self):        self.assertFalse(config.safe_tuple_eq(('a', 'b'), ('a', 'c')))
    def test_diff_length(self):    self.assertFalse(config.safe_tuple_eq(('a',), ('a', 'b')))
    def test_na_vs_empty(self):    self.assertTrue(config.safe_tuple_eq((None, 'b'), ('', 'b')))
    def test_pdna_vs_none(self):   self.assertTrue(config.safe_tuple_eq((pd.NA,), (None,)))
    def test_nan_vs_empty(self):   self.assertTrue(config.safe_tuple_eq((np.nan,), ('',)))
    def test_whitespace(self):     self.assertTrue(config.safe_tuple_eq(('  a  ',), ('a',)))
    def test_both_empty(self):     self.assertTrue(config.safe_tuple_eq((), ()))

In [0]:
# ============================================================
# TEST: to_audit_str() – audit string normalization
# ============================================================

class TestToAuditStr(unittest.TestCase):
    def test_none(self):       self.assertIsNone(config.to_audit_str(None))
    def test_pd_na(self):      self.assertIsNone(config.to_audit_str(pd.NA))
    def test_float_nan(self):  self.assertIsNone(config.to_audit_str(float('nan')))
    def test_string(self):     self.assertEqual(config.to_audit_str('hello'), 'hello')
    def test_number(self):     self.assertEqual(config.to_audit_str(42), '42')

    def test_timestamp(self):
        ts = pd.Timestamp('2024-01-15 10:30:00')
        self.assertIn('2024-01-15', config.to_audit_str(ts))

    def test_datetime(self):
        dt = datetime(2024, 1, 15, 10, 30)
        self.assertIn('2024-01-15', config.to_audit_str(dt))

In [0]:
# ============================================================
# TEST: build_composite_where() + build_composite_set()
# ============================================================

class TestBuildCompositeWhere(unittest.TestCase):
    def test_basic(self):
        row = pd.Series({'col1': 'val1', 'col2': 42})
        clause, params = config.build_composite_where(row, ['col1', 'col2'])
        self.assertEqual(clause, 'col1 = ? AND col2 = ?')
        self.assertEqual(params, ['val1', 42])

    def test_null_value(self):
        row = pd.Series({'col1': 'val1', 'col2': None})
        clause, params = config.build_composite_where(row, ['col1', 'col2'])
        self.assertIn('col2 IS NULL', clause)
        self.assertEqual(len(params), 1)

    def test_timestamp(self):
        ts = pd.Timestamp('2024-01-15 10:30:00')
        row = pd.Series({'ts_col': ts})
        clause, params = config.build_composite_where(row, ['ts_col'])
        self.assertEqual(clause, 'ts_col = ?')
        self.assertIsInstance(params[0], datetime)

    def test_single_column(self):
        row = pd.Series({'id': 1})
        clause, params = config.build_composite_where(row, ['id'])
        self.assertEqual(clause, 'id = ?')
        self.assertEqual(params, [1])


class TestBuildCompositeSet(unittest.TestCase):
    def test_basic(self):
        row = pd.Series({'col1': 'new', 'col2': 99})
        clause, params = config.build_composite_set(row, ['col1', 'col2'])
        self.assertEqual(clause, 'col1 = ?, col2 = ?')
        self.assertEqual(params, ['new', 99])

    def test_null_value(self):
        row = pd.Series({'col1': 'val', 'col2': None})
        clause, params = config.build_composite_set(row, ['col1', 'col2'])
        self.assertIn('col2 = NULL', clause)
        self.assertEqual(len(params), 1)

In [0]:
# ============================================================
# TEST: admin_get_visible_columns()
# ============================================================

class TestAdminGetVisibleColumns(unittest.TestCase):
    def test_no_config(self):
        result = config.admin_get_visible_columns('unknown@test.com', 'Master Table', ['a', 'b', 'c'])
        self.assertEqual(result, ['a', 'b', 'c'])

    def test_wildcard(self):
        config.ADMIN_COLUMN_ACCESS = {'user@test.com': {'Master Table': ['*']}}
        result = config.admin_get_visible_columns('user@test.com', 'Master Table', ['a', 'b', 'c'])
        self.assertEqual(result, ['a', 'b', 'c'])

    def test_restricted(self):
        config.ADMIN_COLUMN_ACCESS = {'user@test.com': {'Master Table': ['a', 'c']}}
        result = config.admin_get_visible_columns('user@test.com', 'Master Table', ['a', 'b', 'c'])
        self.assertEqual(result, ['a', 'c'])

    def test_different_table(self):
        config.ADMIN_COLUMN_ACCESS = {'user@test.com': {'Mapping Codes': ['a']}}
        result = config.admin_get_visible_columns('user@test.com', 'Master Table', ['a', 'b'])
        self.assertEqual(result, ['a', 'b'])

    def test_case_insensitive(self):
        config.ADMIN_COLUMN_ACCESS = {'user@test.com': {'Master Table': ['a']}}
        result = config.admin_get_visible_columns('User@Test.com', 'Master Table', ['a', 'b'])
        self.assertEqual(result, ['a'])

    def tearDown(self):
        config.ADMIN_COLUMN_ACCESS = {}

In [0]:
# ============================================================
# TEST: get_disabled_columns() – per-user editable overrides
# ============================================================

class TestGetDisabledColumns(unittest.TestCase):
    def setUp(self):
        self._orig = config.DISABLED_COLUMNS[:]
        self._orig_access = dict(config.EDITOR_COLUMN_ACCESS)

    def test_default_all_disabled(self):
        config.DISABLED_COLUMNS = ['sp_test_id', 'cl_test_id', 'ingestion_timestamp']
        config.EDITOR_COLUMN_ACCESS = {}
        result = config.get_disabled_columns('anyone@test.com')
        self.assertEqual(result, ['sp_test_id', 'cl_test_id', 'ingestion_timestamp'])

    def test_user_unlocks_one(self):
        config.DISABLED_COLUMNS = ['sp_test_id', 'cl_test_id', 'ingestion_timestamp']
        config.EDITOR_COLUMN_ACCESS = {'editor@test.com': ['cl_test_id']}
        result = config.get_disabled_columns('editor@test.com')
        self.assertEqual(result, ['sp_test_id', 'ingestion_timestamp'])

    def test_case_insensitive(self):
        config.DISABLED_COLUMNS = ['sp_test_id', 'cl_test_id']
        config.EDITOR_COLUMN_ACCESS = {'editor@test.com': ['cl_test_id']}
        result = config.get_disabled_columns('Editor@Test.com')
        self.assertEqual(result, ['sp_test_id'])

    def test_user_unlocks_all(self):
        config.DISABLED_COLUMNS = ['sp_test_id', 'cl_test_id']
        config.EDITOR_COLUMN_ACCESS = {'power@test.com': ['sp_test_id', 'cl_test_id']}
        result = config.get_disabled_columns('power@test.com')
        self.assertEqual(result, [])

    def tearDown(self):
        config.DISABLED_COLUMNS = self._orig
        config.EDITOR_COLUMN_ACCESS = self._orig_access

In [0]:
# ============================================================
# TEST: is_user_in_editor_group() – AAD group via SCIM
# ============================================================

class TestEditorGroupSCIM(unittest.TestCase):

    def _call_with_mock_ws(self, user_email, mock_ws_instance):
        config.EDITOR_GROUPS = ['AAD-EDITORS']
        config.EDITOR_USERS = []
        mock_ws_cls = MagicMock(return_value=mock_ws_instance)
        with patch.dict('sys.modules', {'databricks.sdk': MagicMock(WorkspaceClient=mock_ws_cls)}):
            return config.is_user_in_editor_group(user_email)

    def test_user_is_member(self):
        ws = MagicMock(); u = MagicMock(); u.groups = [MagicMock(display='AAD-EDITORS')]
        ws.users.list.return_value = iter([u])
        self.assertTrue(self._call_with_mock_ws('editor@test.com', ws))

    def test_user_not_member(self):
        ws = MagicMock(); u = MagicMock(); u.groups = [MagicMock(display='Other')]
        ws.users.list.return_value = iter([u])
        self.assertFalse(self._call_with_mock_ws('viewer@test.com', ws))

    def test_user_not_found(self):
        ws = MagicMock(); ws.users.list.return_value = iter([])
        self.assertFalse(self._call_with_mock_ws('ghost@test.com', ws))

    def test_user_no_groups(self):
        ws = MagicMock(); u = MagicMock(); u.groups = None
        ws.users.list.return_value = iter([u])
        self.assertFalse(self._call_with_mock_ws('nogroups@test.com', ws))

    def test_scim_fails_open(self):
        config.EDITOR_GROUPS = ['AAD-EDITORS']; config.EDITOR_USERS = []
        mock_cls = MagicMock(side_effect=Exception('SCIM unavailable'))
        with patch.dict('sys.modules', {'databricks.sdk': MagicMock(WorkspaceClient=mock_cls)}):
            self.assertTrue(config.is_user_in_editor_group('anyone@test.com'))

    def test_user_in_multiple_groups(self):
        ws = MagicMock(); u = MagicMock()
        u.groups = [MagicMock(display='X'), MagicMock(display='AAD-EDITORS'), MagicMock(display='Y')]
        ws.users.list.return_value = iter([u])
        self.assertTrue(self._call_with_mock_ws('multi@test.com', ws))

    def test_no_group_everyone_edits(self):
        config.EDITOR_GROUPS = []; config.EDITOR_USERS = []
        self.assertTrue(config.is_user_in_editor_group('anyone@test.com'))

    def test_editor_users_priority(self):
        """EDITOR_USERS takes priority – user in list always gets write."""
        config.EDITOR_GROUPS = ['AAD-EDITORS']; config.EDITOR_USERS = ['vip@test.com']
        self.assertTrue(config.is_user_in_editor_group('vip@test.com'))

    def test_editor_users_not_in_list(self):
        config.EDITOR_GROUPS = []; config.EDITOR_USERS = ['vip@test.com']
        self.assertFalse(config.is_user_in_editor_group('other@test.com'))

    def test_multi_groups_match(self):
        """User in second configured group should pass."""
        ws = MagicMock(); u = MagicMock()
        u.groups = [MagicMock(display='GROUP-B')]
        ws.users.list.return_value = iter([u])
        config.EDITOR_GROUPS = ['GROUP-A', 'GROUP-B']
        config.EDITOR_USERS = []
        mock_ws_cls = MagicMock(return_value=ws)
        with patch.dict('sys.modules', {'databricks.sdk': MagicMock(WorkspaceClient=mock_ws_cls)}):
            self.assertTrue(config.is_user_in_editor_group('user@test.com'))

    def tearDown(self):
        config.EDITOR_GROUPS = []; config.EDITOR_USERS = []

In [0]:
# ============================================================
# TEST: dropdown_options()
# ============================================================

class TestDropdownOptions(unittest.TestCase):
    def test_with_cache(self):
        config._dropdown_cache = {'col': ['A', 'B', 'C']}
        result = config.dropdown_options('col', [])
        self.assertIn('A', result)
        self.assertIn('TBC', result)
        self.assertEqual(result, sorted(result))

    def test_fallback_to_db(self):
        config._dropdown_cache = {'col': []}
        result = config.dropdown_options('col', ['X', 'Y', None])
        self.assertIn('X', result)
        self.assertIn('Y', result)
        self.assertNotIn('None', result)

    def test_unknown_col(self):
        config._dropdown_cache = {}
        result = config.dropdown_options('unknown', ['P', 'Q'])
        self.assertIn('P', result)
        self.assertIn('TBC', result)

    def test_tbc_not_duplicated(self):
        config._dropdown_cache = {'col': ['TBC', 'A']}
        self.assertEqual(config.dropdown_options('col', []).count('TBC'), 1)

    def test_sorted(self):
        config._dropdown_cache = {'col': ['Z', 'A', 'M']}
        result = config.dropdown_options('col', [])
        self.assertEqual(result, sorted(result))

In [0]:
# ============================================================
# TEST: generate_test_id() – SP test ID generation
# ============================================================

class TestGenerateTestId(unittest.TestCase):
    def test_basic_format(self):
        with patch.object(config, 'get_prod_catL1_code', return_value='CAT01'):
            with patch.object(config, 'get_brand_L0_code', return_value='BRD01'):
                result = config.generate_test_id('ST', 2024, 'France', 'Yogurt', 'Danone', None, 5)
                self.assertEqual(result, 'SP_2024_France_CAT01_BRD01_5_N')

    def test_retest_yes(self):
        with patch.object(config, 'get_prod_catL1_code', return_value='C1'):
            with patch.object(config, 'get_brand_L0_code', return_value='B1'):
                result = config.generate_test_id('ST', 2024, 'UK', 'Cat', 'Brand', '2025', 3)
                self.assertTrue(result.endswith('_Y'))

    def test_retest_no(self):
        with patch.object(config, 'get_prod_catL1_code', return_value='C1'):
            with patch.object(config, 'get_brand_L0_code', return_value='B1'):
                result = config.generate_test_id('ST', 2024, 'UK', 'Cat', 'Brand', '', 3)
                self.assertTrue(result.endswith('_N'))

    def test_missing_values_use_NA(self):
        with patch.object(config, 'get_prod_catL1_code', return_value=None):
            with patch.object(config, 'get_brand_L0_code', return_value=None):
                result = config.generate_test_id(None, None, None, None, None, None, 1)
                self.assertIn('NA', result)

    def test_float_year_no_decimal(self):
        with patch.object(config, 'get_prod_catL1_code', return_value='C'):
            with patch.object(config, 'get_brand_L0_code', return_value='B'):
                result = config.generate_test_id('ST', 2024.0, 'FR', 'X', 'Y', None, 1)
                self.assertIn('2024', result)
                self.assertNotIn('.0', result)

In [0]:
# ============================================================
# TEST: build_audit_events()
# ============================================================

class TestBuildAuditEvents(unittest.TestCase):
    def setUp(self):
        mock_st.session_state = {'session_id': 'test-session-123'}

    def test_insert_event(self):
        orig = pd.DataFrame({'col1': ['a']}).set_index(pd.Index([1], name='row_id'))
        added = pd.DataFrame({'col1': ['new']})
        with patch.object(config, 'get_user_identity', return_value='test@user.com'):
            events = config.build_audit_events(
                original_df=orig, edited_df=orig.copy(),
                added_rows_df=added, deleted_rows_df=None, diff_df=None, page_no=1)
        self.assertIn('INSERT_ROW', [e['event_type'] for e in events])

    def test_delete_event(self):
        orig = pd.DataFrame({'col1': ['a']}).set_index(pd.Index([1], name='row_id'))
        deleted = pd.DataFrame({'col1': ['del']}).set_index(pd.Index([2], name='row_id'))
        with patch.object(config, 'get_user_identity', return_value='test@user.com'):
            events = config.build_audit_events(
                original_df=orig, edited_df=orig.copy(),
                added_rows_df=None, deleted_rows_df=deleted, diff_df=None,
                page_no=1, deleted_row_ids=[2])
        self.assertIn('DELETE_ROW', [e['event_type'] for e in events])

    def test_custom_source(self):
        orig = pd.DataFrame({'col1': ['a']}).set_index(pd.Index([1], name='row_id'))
        with patch.object(config, 'get_user_identity', return_value='admin@user.com'):
            events = config.build_audit_events(
                original_df=orig, edited_df=orig.copy(),
                added_rows_df=None, deleted_rows_df=None, diff_df=None,
                page_no=1, source='Admin UI', session_key='admin_session_id')
        save_evt = [e for e in events if e['event_type'] == 'SAVE'][0]
        self.assertIn('Admin UI', save_evt['notes'])

    def test_empty_changes_still_has_save(self):
        orig = pd.DataFrame({'col1': ['a']}).set_index(pd.Index([1], name='row_id'))
        with patch.object(config, 'get_user_identity', return_value='test@user.com'):
            events = config.build_audit_events(
                original_df=orig, edited_df=orig.copy(),
                added_rows_df=None, deleted_rows_df=None, diff_df=None, page_no=1)
        self.assertEqual(len(events), 1)
        self.assertEqual(events[0]['event_type'], 'SAVE')

In [0]:
# ============================================================
# TEST: Constants + ensure_session_id()
# ============================================================

class TestConstants(unittest.TestCase):
    def test_table_fqn_3_parts(self):    self.assertEqual(len(config.TABLE_FQN.split('.')), 3)
    def test_select_cols_has_row_id(self): self.assertIn('row_id', config.SELECT_COLUMNS)
    def test_dml_cols_no_row_id(self):    self.assertNotIn('row_id', config.DML_COLUMNS)
    def test_schema_dtype_complete(self):
        for col in config.SELECT_COLUMNS:
            self.assertIn(col, config.SCHEMA_DTYPE, f'{col} missing from SCHEMA_DTYPE')
    def test_upload_dtype_no_row_id(self): self.assertNotIn('row_id', config.UPLOAD_DTYPE)
    def test_disabled_cols(self):          self.assertIn('sp_test_id', config.DISABLED_COLUMNS)
    def test_mandatory_cols(self):         self.assertIn('test_year', config.MANDATORY_UPDATE_COLS)
    def test_page_size_positive(self):     self.assertGreater(config.PAGE_SIZE, 0)
    def test_css_has_style_tag(self):      self.assertIn('<style>', config.SHARED_CSS)
    def test_admin_tables_map(self):
        for key in ['Master Table', 'Mapping Codes', 'Dropdown Options']:
            self.assertIn(key, config.ADMIN_TABLES)


class TestEnsureSessionId(unittest.TestCase):
    def test_creates_new(self):
        mock_st.session_state = {}
        config.ensure_session_id('test_key')
        self.assertIn('test_key', mock_st.session_state)
        self.assertEqual(len(mock_st.session_state['test_key']), 36)

    def test_preserves_existing(self):
        mock_st.session_state = {'test_key': 'existing-id'}
        config.ensure_session_id('test_key')
        self.assertEqual(mock_st.session_state['test_key'], 'existing-id')

    def test_default_key(self):
        mock_st.session_state = {}
        config.ensure_session_id()
        self.assertIn('session_id', mock_st.session_state)

In [0]:
# ============================================================
# RUN ALL TESTS
# ============================================================

loader = unittest.TestLoader()
suite = unittest.TestSuite()

test_classes = [
    TestToBold, TestIsNa, TestValidateMandatoryCols,
    TestNormValue, TestSafeTupleEq, TestToAuditStr,
    TestBuildCompositeWhere, TestBuildCompositeSet,
    TestAdminGetVisibleColumns, TestGetDisabledColumns,
    TestEditorGroupSCIM, TestDropdownOptions,
    TestGenerateTestId, TestBuildAuditEvents,
    TestConstants, TestEnsureSessionId,
]

for cls in test_classes:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

passed = result.testsRun - len(result.failures) - len(result.errors)
print(f'\n{"="*60}')
print(f'  TOTAL: {result.testsRun}  |  PASSED: {passed}  |  FAILED: {len(result.failures)}  |  ERRORS: {len(result.errors)}')
print(f'{"="*60}')

if result.wasSuccessful():
    print('\n✅ ALL TESTS PASSED')
else:
    print('\n❌ SOME TESTS FAILED')